# A.5 答錯是沒找到還是沒用好？

# A：上下文學習與 RAG

前置：第7章prompt與對話。不需要圖片或MoE。把模型想成可以讀你遞給它的小抄的答題者：先手動給例子，再從本地文件找相關段落。檢索成功不等於回答一定正確。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：像先找小抄，再核對答案來源**

找到對文件與用對資料，是兩個不同分數。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rag.svg")))

**先想一想：**回答錯誤全都該怪檢索嗎？

錯誤可能發生在沒找到文件，也可能找到卻沒用對。像考試帶對小抄還抄錯位置；兩個階段要各自量。

**把直覺寫成數學：**retrieval_recall=含正確來源題數/總題數；answer_accuracy另外計算。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
examples = [{"hit": True, "correct": True}, {"hit": True, "correct": False}, {"hit": False, "correct": False}]
print("retrieval recall", sum(e["hit"] for e in examples) / len(examples))
print("answer accuracy", sum(e["correct"] for e in examples) / len(examples))

**如何讀結果：**人工構造三例說明分解，真評估從保存的retrieval trace與答案得到。

**只改一件事：**只修正第二例回答，覈對retrieval分數不變。
